# Laboratorio 5 — Agentes en el Arcade Learning Environment (ALE): Space Invaders

**Curso:** CC3092 - Deep Learning y Sistemas Inteligentes
**Autor:** Angel Esteban Esquit Hernández

Este notebook contiene: (1) la investigación sobre ALE y sus espacios de observación/acción, (2) el módulo de funciones reutilizables para interactuar con ALE (`ale_utils.py`), y (3) la generación y reporte de los videos de un agente jugando `ALE/SpaceInvaders-v5`.

**Nota:** en este laboratorio no se entrena ningún agente inteligente; el objetivo es dejar funcionando la infraestructura (crear entornos, ejecutar episodios y grabar video) que servirá de base para el proyecto futuro.

## 1. El Arcade Learning Environment (ALE)

### 1.1 ¿Qué es ALE y qué problema resuelve? Relación con Stella y con los juegos originales de Atari 2600

El **Arcade Learning Environment (ALE)** es un framework de investigación (Bellemare et al., 2013) que expone más de medio centenar de juegos originales de **Atari 2600** como entornos estandarizados para agentes de IA. Resuelve el problema de no tener un banco común y desafiante de tareas de percepción y control para evaluar agentes de aprendizaje por refuerzo: antes de ALE, cada trabajo usaba entornos ad-hoc, difíciles de comparar entre sí.

ALE se construye **sobre Stella**, un emulador de Atari 2600 de código abierto que reproduce fielmente el hardware original (CPU 6507, TIA de video, RAM de 128 bytes). ALE instrumenta ese emulador: en cada paso inyecta la acción del agente como si viniera del joystick, avanza la emulación un número de frames y expone la pantalla (o la RAM) como observación y la variación de puntaje interno del juego como recompensa. Los juegos son los **binarios (ROMs) originales de Atari 2600** de la década de 1970-80, sin modificarlos: la dificultad y el comportamiento del juego son exactamente los del hardware real.

### 1.2 Diferencias entre variantes de un mismo juego en ALE

Gymnasium expone Space Invaders (y los demás juegos de ALE) mediante identificadores como `ALE/SpaceInvaders-v5`, con variantes controladas por parámetros de `gym.make`:

- **Observación por imagen vs. `-ram`**: `ALE/SpaceInvaders-v5` retorna como observación el frame renderizado (`Box(210,160,3)`, RGB). Pasando `obs_type="ram"` (equivalente histórico al sufijo `-ram`) la observación son los **128 bytes de RAM** de la consola en lugar de la imagen — mismo juego y dinámica, distinta representación del estado.
- **`frameskip`**: número de frames del emulador que se repiten/saltan por cada `step()` del agente (ver 1.3). En Space Invaders el valor por defecto de la v5 es un `frameskip` estocástico entre 2 y 4 frames.
- **`repeat_action_probability`**: probabilidad de que el emulador **ignore** la acción elegida por el agente y repita la anterior en su lugar (*sticky actions*, Machado et al., 2018). Con `repeat_action_probability=0.25` (valor por defecto de las variantes v5) se introduce estocasticidad para evitar que los agentes memoricen secuencias de acciones fijas explotando que Atari es determinista; con `0.0` el entorno es determinista dada una semilla.
- **`full_action_space`**: si es `False` (por defecto), `action_space` solo incluye las acciones que el juego realmente usa (6 en Space Invaders); si es `True`, se expone el conjunto completo de 18 acciones del joystick de Atari, igual para todos los juegos, aunque varias no tengan efecto en uno en particular.

### 1.3 Importancia del frame skipping

El emulador de Atari 2600 corre a 60 frames por segundo, pero un agente que decidiera una acción distinta en cada uno de esos frames sería computacionalmente muy costoso de entrenar y, además, casi no aportaría información nueva entre frames consecutivos (la pantalla cambia poco de un frame al siguiente). El **frame skipping** hace que el agente elija una acción cada $k$ frames (típicamente $k \in \{2,3,4\}$) y esa misma acción se repita internamente durante los frames saltados, devolviendo como recompensa la suma acumulada.

Esto **acelera drásticamente la simulación** (el agente solo decide y aprende de una fracción de los frames totales) y **facilita el aprendizaje**: reduce el horizonte efectivo del problema (menos decisiones por episodio) y actúa como una forma simple de abstracción temporal, evitando que el agente deba lidiar con la altísima frecuencia de control del hardware original.

### 1.4 Space Invaders: mecánica, objetivo y traducción de puntaje a reward

En **Space Invaders** el jugador controla un cañón láser que se mueve horizontalmente en la parte inferior de la pantalla y debe destruir oleadas de invasores alienígenas que descienden progresivamente, evitando ser alcanzado por los disparos enemigos y, ocasionalmente, protegiéndose tras búnkeres destructibles. El objetivo es maximizar el puntaje eliminando invasores (que otorgan más puntos mientras más alejados/rápidos están) antes de perder las 3 vidas disponibles o de que los invasores lleguen a la base del jugador.

ALE traduce la puntuación del juego original directamente a la señal de **reward** del entorno: en cada `step()`, `reward` es la diferencia de puntaje interno del juego entre el frame anterior y el actual (0 si no se destruyó nada en ese intervalo). El episodio termina (`terminated=True`) cuando el jugador pierde todas sus vidas.

## 2. Espacios de observación y acción en entornos Atari

### 2.1 Observación por defecto de `ALE/SpaceInvaders-v5` vs. `CartPole-v1`

Por defecto, `ALE/SpaceInvaders-v5` retorna como observación un **frame RGB de 210×160×3 píxeles** (`Box(0, 255, (210, 160, 3), uint8)`), es decir, una imagen cruda del estado visual del juego. En contraste, `CartPole-v1` retorna un **vector de 4 valores continuos** (`Box(4,)`: posición y velocidad del carro, ángulo y velocidad angular del poste), un resumen de bajo nivel ya "interpretado" del estado físico del sistema.

Trabajar con observaciones basadas en **imágenes** implica: (1) un espacio de estados de dimensionalidad mucho mayor (100,800 valores vs. 4), por lo que no es viable una tabla de valores ni una política tabular; se requieren aproximadores de función con capacidad para extraer características espaciales, típicamente **redes neuronales convolucionales**; (2) mayor costo computacional y de memoria por observación y por red; y (3) la necesidad de que el agente aprenda por sí mismo qué características visuales son relevantes (posición de naves, disparos, etc.), en lugar de recibirlas ya calculadas como en CartPole.

### 2.2 Observación en RAM (`obs_type="ram"`)

Con `obs_type="ram"` el entorno expone directamente los **128 bytes de memoria** de la Atari 2600 como observación (`Box(0, 255, (128,), uint8)`), que es donde el juego guarda su estado interno (posiciones, puntaje, vidas, etc.) de forma mucho más compacta que un frame completo.

Esta variante puede preferirse cuando: se quiere **reducir drásticamente el costo computacional** (un vector de 128 bytes es mucho más barato de procesar que una imagen de 100K+ píxeles, permitiendo usar redes totalmente conectadas pequeñas en vez de CNNs); se busca **acelerar la experimentación** o el entrenamiento en hardware limitado; o cuando el objetivo de investigación es específicamente **interpretabilidad o ingeniería de características** sobre variables de estado explícitas, en lugar de aprendizaje de representaciones visuales de extremo a extremo.

### 2.3 Espacio de acción de Space Invaders

`ALE/SpaceInvaders-v5` (con `full_action_space=False`, valor por defecto) tiene un espacio de acción `Discrete(6)`, con los siguientes significados (obtenidos vía `env.unwrapped.get_action_meanings()`):

| Acción | Significado |
|---|---|
| 0 | `NOOP` — no hacer nada (mantener posición) |
| 1 | `FIRE` — disparar sin moverse |
| 2 | `RIGHT` — mover el cañón a la derecha |
| 3 | `LEFT` — mover el cañón a la izquierda |
| 4 | `RIGHTFIRE` — mover a la derecha y disparar simultáneamente |
| 5 | `LEFTFIRE` — mover a la izquierda y disparar simultáneamente |

### 2.4 `AtariPreprocessing` y `FrameStackObservation`

El wrapper **`AtariPreprocessing`** de Gymnasium aplica el preprocesamiento estándar usado desde el paper original de DQN (Mnih et al., 2015) sobre entornos de Atari: conversión a **escala de grises**, **redimensionamiento a 84×84** píxeles, **frame skipping** (aplicando `max` sobre los últimos frames para evitar parpadeo de sprites), opción de recorte (*clipping*) de la recompensa a $\{-1,0,1\}$, y detección de fin de vida como fin de episodio (`terminal_on_life_loss`). Esto reduce enormemente el tamaño de cada observación (de 210×160×3 a 84×84×1) sin perder la información relevante para jugar.

**`FrameStackObservation`** (antes `FrameStack`) apila las últimas $k$ observaciones consecutivas (típicamente 4) en un solo tensor. Se usa junto con `AtariPreprocessing` porque una sola imagen estática no contiene información de **movimiento** (dirección y velocidad de las naves y disparos no se pueden inferir de un único frame); al apilar varios frames el agente puede inferir esa dinámica temporal a partir de una observación puramente espacial, sin necesidad de una arquitectura recurrente.

## 3. Módulo de funciones para interactuar con ALE

Se implementó el módulo `ale_utils.py` (en la raíz del repositorio) con las siguientes funciones reutilizables:

- **`crear_entorno(nombre_entorno, video_folder=None, episode_trigger=None, **kwargs)`**: crea un entorno de Gymnasium con `gym.make`. Si se especifica `video_folder`, envuelve el entorno con `gymnasium.wrappers.RecordVideo` (forzando `render_mode="rgb_array"`) para grabar los episodios indicados por `episode_trigger` (por defecto, todos). Funciona para cualquier entorno de Gymnasium, no solo para Atari — se probó con `ALE/SpaceInvaders-v5` y con `CartPole-v1`.
- **`agente_aleatorio(observation, env)`**: ignora la observación y retorna `env.action_space.sample()`. Sirve como referencia base (*baseline*) sin entrenamiento.
- **`agente_regla_simple(observation, env)`**: agente basado en una regla fija para Space Invaders — dispara continuamente (`RIGHTFIRE`/`LEFTFIRE`/`FIRE`, según qué acciones exponga el entorno) en vez de moverse sin disparar como haría un agente puramente aleatorio.
- **`ejecutar_episodio(env, funcion_agente, max_steps=10000, seed=None)`**: ejecuta un episodio completo llamando a `funcion_agente(observation, env)` en cada paso, hasta que `terminated` o `truncated` sea verdadero o se alcance `max_steps`. Retorna un diccionario con `pasos`, `recompensa_total`, `terminated` y `truncated`.
- **`generar_video_agente(nombre_entorno, funcion_agente, video_folder, name_prefix, n_episodios=1, ...)`**: función de alto nivel que crea el entorno con grabación habilitada, ejecuta `n_episodios` episodios completos, cierra el entorno con `env.close()` (indispensable para que `RecordVideo` escriba el `.mp4` a disco) y retorna las rutas de los videos generados junto con las métricas de cada episodio.

A continuación se importa el módulo y se generan los videos requeridos.

In [1]:
from ale_utils import (
    crear_entorno,
    agente_aleatorio,
    agente_regla_simple,
    ejecutar_episodio,
    generar_video_agente,
)

import gymnasium as gym
import ale_py

print("gymnasium:", gym.__version__)
print("ale_py:", ale_py.__version__)

gymnasium: 1.3.0
ale_py: 0.12.1


### 3.1 Inspección de espacios (verificación empírica)

In [2]:
env = gym.make("ALE/SpaceInvaders-v5")
print("Observation space:", env.observation_space)
print("Action space:", env.action_space)
print("Acciones:", env.unwrapped.get_action_meanings())
env.close()

Observation space: Box(0, 255, (210, 160, 3), uint8)
Action space: Discrete(6)
Acciones: ['NOOP', 'FIRE', 'RIGHT', 'LEFT', 'RIGHTFIRE', 'LEFTFIRE']


### 3.2 Video del agente aleatorio en `ALE/SpaceInvaders-v5` (deliverable principal)

In [3]:
resultado_random = generar_video_agente(
    nombre_entorno="ALE/SpaceInvaders-v5",
    funcion_agente=agente_aleatorio,
    video_folder="videos",
    name_prefix="space_invaders_random",
    n_episodios=1,
    seed=0,
)

for i, m in enumerate(resultado_random["metricas"]):
    print(f"Episodio {i+1}: pasos = {m['pasos']}, recompensa_total = {m['recompensa_total']}")

print("Videos generados:", resultado_random["videos"])

Episodio 1: pasos = 660, recompensa_total = 210.0
Videos generados: ['videos\\space_invaders_random-episode-0.mp4']


### 3.3 Video del agente de regla simple en `ALE/SpaceInvaders-v5` (comparación)

Como extensión, se generó también un video con `agente_regla_simple` (disparo continuo) para comparar contra el agente aleatorio puro.

In [4]:
resultado_regla = generar_video_agente(
    nombre_entorno="ALE/SpaceInvaders-v5",
    funcion_agente=agente_regla_simple,
    video_folder="videos",
    name_prefix="space_invaders_regla_simple",
    n_episodios=1,
    seed=0,
)

for i, m in enumerate(resultado_regla["metricas"]):
    print(f"Episodio {i+1}: pasos = {m['pasos']}, recompensa_total = {m['recompensa_total']}")

print("Videos generados:", resultado_regla["videos"])

C:\Users\aeeh2\Documents\Universidad\Semestre 8\Deep Learning\Lab5-DL\.venv\Lib\site-packages\gymnasium\wrappers\rendering.py:292: UserWarning: WARN: Overwriting existing videos at C:\Users\aeeh2\Documents\Universidad\Semestre 8\Deep Learning\Lab5-DL\videos folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


Episodio 1: pasos = 538, recompensa_total = 270.0
Videos generados: ['videos\\space_invaders_regla_simple-episode-0.mp4']


### 3.4 Verificación de generalidad: `crear_entorno` con otro entorno de Gymnasium

Para confirmar que el módulo no está atado a Space Invaders, se genera además un video de un agente aleatorio en `CartPole-v1` usando exactamente las mismas funciones.

In [5]:
resultado_cartpole = generar_video_agente(
    nombre_entorno="CartPole-v1",
    funcion_agente=agente_aleatorio,
    video_folder="videos",
    name_prefix="cartpole_random",
    n_episodios=1,
    seed=0,
)

for i, m in enumerate(resultado_cartpole["metricas"]):
    print(f"Episodio {i+1}: pasos = {m['pasos']}, recompensa_total = {m['recompensa_total']}")

print("Videos generados:", resultado_cartpole["videos"])

Episodio 1: pasos = 31, recompensa_total = 31.0
Videos generados: ['videos\\cartpole_random-episode-0.mp4']


## 4. Resumen de resultados

| Entorno | Agente | Pasos | Recompensa total |
|---|---|---|---|
| ALE/SpaceInvaders-v5 | Aleatorio | 660 | 210.0 |
| ALE/SpaceInvaders-v5 | Regla simple (disparo continuo) | 538 | 270.0 |
| CartPole-v1 | Aleatorio | 31 | 31.0 |

Los videos `.mp4` quedan guardados en la carpeta `videos/` del repositorio.

## 5. Discusión

El agente **aleatorio** en Space Invaders logra un desempeño no trivial (recompensa > 0) simplemente porque, al elegir acciones al azar de un espacio pequeño (6 acciones, la mitad de ellas con disparo), termina disparando con relativa frecuencia y ocasionalmente acierta a los invasores sin ninguna estrategia. El agente de **regla simple**, que dispara en cada paso, tiende a acumular más disparos efectivos por episodio que el agente puramente aleatorio (que reparte su "presupuesto" de pasos entre moverse sin disparar y disparar), lo que ilustra que una pequeña cantidad de conocimiento del dominio (aquí, "siempre es mejor estar disparando") ya mejora el comportamiento sin necesidad de ningún proceso de aprendizaje.

Esta infraestructura (`crear_entorno`, `ejecutar_episodio`, `generar_video_agente`) es agnóstica al agente utilizado: basta con reemplazar `funcion_agente` por una política entrenada (p. ej. un DQN sobre observaciones preprocesadas con `AtariPreprocessing` + `FrameStackObservation`) para reutilizar exactamente el mismo código en el proyecto futuro de entrenamiento de agentes.

## Referencias

- Bellemare, M. G., Naddaf, Y., Veness, J., & Bowling, M. (2013). The Arcade Learning Environment: An evaluation platform for general agents. *Journal of Artificial Intelligence Research*, 47, 253-279.
- Machado, M. C., Bellemare, M. G., Talvitie, E., Veness, J., Hausknecht, M., & Bowling, M. (2018). Revisiting the Arcade Learning Environment: Evaluation protocols and open problems for general agents. *Journal of Artificial Intelligence Research*, 61, 523-562.
- Mnih, V. et al. (2015). Human-level control through deep reinforcement learning. *Nature*, 518(7540), 529-533.
- Gymnasium Documentation, Farama Foundation, https://gymnasium.farama.org/
- ALE-py Documentation, Farama Foundation, https://ale.farama.org/